# COCO neural channel — VideoSeal download recovery

Resume the current **100-image run**, with the same bank, output folder, batch size and A100-SXM4-80GB execution environment. Do not remove saved results or attack ZIPs.

Only VideoSeal checkpoint transport changes: Meta's endpoint has a pinned mirror fallback. Bytes are accepted only if their SHA-256 matches the saved bank, then cached on Drive. The previous SDXL identity-order repair is retained unchanged. All scientific checks remain enabled.

Run all cells in order in a **separate runtime from any active digital experiment**. The unchanged source ZIP is reused from Drive. CPU/static tests do not replace a real-GPU resume acceptance run.


## 0. Configuration

Change only `N_IMAGES` from 5 to 100 after reviewing the canary. Keep the same output folder and batch size.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile

N_IMAGES = 100                # Resume the current full run; use 5 only for the earlier canary.
MODELS = "all"                # All 29 reviewed production entries.
GPU_BATCH_SIZE = 4            # Freeze this after the first canary; A100 high RAM recommended.
MAX_HOURS = 16                # Session budget; not a guarantee of Colab uptime.
SSD_RESERVE_GIB = 60
CAPTIONS_FILE = None          # Optional official captions_val2017.json; otherwise downloaded once.
SOURCE_ARCHIVE = Path("COCO_Neural_Channel_RESUMABLE_source.zip")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Use Colab or Linux/NVIDIA; the complete release cannot run natively on Windows.")
if IN_COLAB:
    drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_production_bank_v1"   # READ ONLY
OUTPUT_ROOT = BASE / "watermarking_neural_channel_v1" # Separate from digital!
SCRATCH_ROOT = Path("/content/wmbench_neural_scratch") if IN_COLAB else Path("/workspace/wmbench_neural_scratch")
SETTINGS = dict(n_images=N_IMAGES, models=MODELS, gpu_batch_size=GPU_BATCH_SIZE, max_hours=MAX_HOURS,
    ssd_reserve_gib=SSD_RESERVE_GIB, device="cuda", bank=str(BANK_ROOT), output=str(OUTPUT_ROOT),
    scratch=str(SCRATCH_ROOT), captions_file=CAPTIONS_FILE, pairwise=True, download_author_assets=True,
    manual_assets_root=str(BANK_ROOT / "_manual_assets"))
print("Input bank (unchanged):", BANK_ROOT)
print("Durable results:", OUTPUT_ROOT)
print("Temporary attack ZIPs on Drive: one method/pipeline stage, deleted after decoding.")
print("Do not use the runtime that is currently executing the digital experiment.")


## 1. Restore the release and isolated controller

No model weights are stored in this small ZIP. Checkpoints download once per Colab session and are shared between methods. Compatible neural pipelines use the controller environment; each watermark decoder uses its own frozen environment.


In [ ]:
EXPECTED_SHA = "5b70653cf2fb7b23d6ff1b8585aa1b684808ba311d956cf5fc20da4b6a6311e2"
SAVED_ZIP = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if not SOURCE_ARCHIVE.is_file() and SAVED_ZIP.is_file():
    SOURCE_ARCHIVE = SAVED_ZIP
if not SOURCE_ARCHIVE.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Place the source ZIP beside this notebook or set SOURCE_ARCHIVE.")
    from google.colab import files
    uploaded = files.upload()
    matches = [Path(n) for n, data in uploaded.items() if hashlib.sha256(data).hexdigest() == EXPECTED_SHA]
    if len(matches) != 1:
        raise ValueError("Upload the matching source ZIP supplied with this notebook.")
    SOURCE_ARCHIVE = matches[0]
data = SOURCE_ARCHIVE.read_bytes()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA:
    raise ValueError("Wrong source ZIP: checksum mismatch")
SAVED_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not SAVED_ZIP.exists():
    SAVED_ZIP.write_bytes(data)
if hashlib.sha256(SAVED_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("Saved source ZIP readback failed")
PROJECT = SCRATCH_ROOT / "release"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or member.is_dir():
            raise ValueError("Unsafe source ZIP entry")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Source changed: {target}; use a different empty scratch, keep results.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
CONTROLLER = SCRATCH_ROOT / "controller/bin/python"
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10.12", str(CONTROLLER.parent.parent)], check=True)
lock = OUTPUT_ROOT / "_release/controller-resolved.lock"
if lock.is_file():
    subprocess.run([UV, "pip", "sync", "--python", str(CONTROLLER), str(lock)], check=True)
else:
    subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER), "-r", str(PROJECT / "requirements.txt")], check=True)
    resolved = subprocess.check_output([UV, "pip", "freeze", "--python", str(CONTROLLER)], text=True)
    lock.write_text(resolved, encoding="utf-8")
    if lock.read_text(encoding="utf-8") != resolved:
        raise ValueError("Resolved environment lock readback failed")
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")]),
    PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="4",
    PATH=str(Path(UV).parent)+os.pathsep+ENV.get("PATH", ""),
    UV_PYTHON_INSTALL_DIR=str(SCRATCH_ROOT / "python"), CUBLAS_WORKSPACE_CONFIG=":4096:8")
CONFIG = SCRATCH_ROOT / "neural-settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print("Free SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


## 1a. Restore the narrow compatibility fix

The readable module below is added beside the original sources; none are overwritten. Its checksum is checked in every worker. Repair provenance is written to `OUTPUT_ROOT/_repairs/scheduler-default-order-v1/`. Old execution manifests are never rewritten.


In [ ]:
# Narrow recovery overlay; all files in the original release remain untouched.
REPAIR_SOURCE = r'''"""Resume release 1.0.0 across permutations of Diffusers bookkeeping keys.

The original release, frozen execution identities, attack code, seeds and result
hashes stay untouched. This entry point delegates every stage to that release.
Only scheduler_config._use_default_values is compared as an unordered set of
unique strings. Diffusers 0.32.2 builds it using list(set(...)); its order does
not select or configure the scheduler's numerical operations.
"""

import argparse
import copy
import hashlib
import json
import os
from pathlib import Path
import sys
from unittest.mock import patch


VERSION = "scheduler-default-order-v1"
BASE_ZIP_SHA256 = "5b70653cf2fb7b23d6ff1b8585aa1b684808ba311d956cf5fc20da4b6a6311e2"
ENTRYPOINT = "neural_sdxl_identity_fix"
HASH_ENV = "WMBENCH_NEURAL_IDENTITY_FIX_SHA256"
DEFAULT_KEYS = "_use_default_values"


def module_sha256():
    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()


def canonical_identity(value):
    """Copy metadata; normalize exactly one field, never the live scheduler."""
    result = copy.deepcopy(value)
    config = result.get("scheduler_config")
    if isinstance(config, dict) and DEFAULT_KEYS in config:
        keys = config[DEFAULT_KEYS]
        if (not isinstance(keys, list) or not all(isinstance(key, str) for key in keys)
                or len(keys) != len(set(keys))):
            raise ValueError("scheduler_config._use_default_values must be unique strings")
        config[DEFAULT_KEYS] = sorted(keys)
    return result


def changed_paths(left, right, prefix=""):
    """Report genuine differences without silently ignoring missing fields."""
    if isinstance(left, dict) and isinstance(right, dict):
        result = []
        for key in sorted(set(left) | set(right)):
            path = f"{prefix}.{key}" if prefix else key
            if key not in left or key not in right:
                result.append(path)
            else:
                result.extend(changed_paths(left[key], right[key], path))
        return result
    return [] if left == right else [prefix]


def reconcile_identity(results, engine, observed):
    """Reuse a legacy identity only after exact semantic-equivalence checking.

    Existing package/record references contain the digest of the legacy JSON.
    Retaining that JSON rather than rewriting it keeps those references valid.
    The actual newly observed metadata is retained in a separate audit receipt.
    """
    from wmbench.io import read_json, sha256
    from wmbench_full_payload.storage import EvidenceError, freeze
    from wmbench_neural.spec import digest

    root = Path(results)
    path = root / "_execution" / f"{engine}.json"
    if not path.exists():
        # A new engine has no historical references to preserve. Its metadata
        # can be deterministic from the outset; original freeze() still runs.
        return canonical_identity(observed)
    saved = read_json(path)
    normalized_saved = canonical_identity(saved)
    normalized_observed = canonical_identity(observed)
    if normalized_saved != normalized_observed:
        differences = changed_paths(normalized_saved, normalized_observed)
        raise EvidenceError(
            f"Execution identity changed for {engine}: {', '.join(differences[:16])}. "
            "Only a permutation of scheduler_config._use_default_values is allowed. "
            "Saved results and execution metadata were not modified."
        )
    if saved != observed:
        receipt = dict(
            repair=VERSION, repair_sha256=module_sha256(),
            base_zip_sha256=BASE_ZIP_SHA256, engine=engine,
            allowed_difference="scheduler_config._use_default_values order only",
            preserved_execution_file=f"_execution/{engine}.json",
            preserved_file_sha256=sha256(path),
            preserved_identity_sha256=digest(saved),
            observed_identity_sha256=digest(observed), observed_identity=observed,
        )
        target = root / "_repairs" / VERSION / f"{engine}-{digest(observed)}.json"
        freeze(target, receipt)
        print(f"IDENTITY REPAIR {engine}: service-key order only; legacy identity "
              f"and results preserved. Audit: {target}", flush=True)
    return saved


def redirect_child(command, env, expected_sha256):
    """Propagate this entry point through process_guard; leave other jobs alone."""
    forwarded = list(command)
    for index in range(len(forwarded) - 1):
        if forwarded[index] == "-m" and forwarded[index + 1] == "wmbench_neural.runner":
            forwarded[index + 1] = ENTRYPOINT
            child_env = dict(env)
            child_env[HASH_ENV] = expected_sha256
            return forwarded, child_env
    return command, env


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("action", choices=("run", "validate", "prepare", "quality",
        "geometry", "controls", "generate", "decode", "report"))
    parser.add_argument("--settings", type=Path, required=True)
    args = parser.parse_args()
    actual_hash = module_sha256()
    if os.environ.get(HASH_ENV) != actual_hash:
        raise RuntimeError("Repair checksum is absent or changed. Rerun the notebook repair cell.")
    settings = json.loads(args.settings.read_text(encoding="utf-8"))
    from wmbench_full_payload.storage import freeze
    from wmbench_neural import runner

    results = settings["results"] if args.action == "generate" else settings.get("output")
    if args.action in {"run", "validate", "generate", "report"}:
        freeze(Path(results) / "_repairs" / VERSION / "repair.json", dict(
            repair=VERSION, repair_sha256=actual_hash, base_zip_sha256=BASE_ZIP_SHA256,
            scope="scheduler bookkeeping-list order only; original release and results unchanged"))
    if args.action == "run":
        from wmbench import colab_canary
        original_monitor = colab_canary.monitored_process

        def monitor(command, env, *positional, **keywords):
            command, env = redirect_child(command, env, actual_hash)
            return original_monitor(command, env, *positional, **keywords)

        with patch.object(colab_canary, "monitored_process", monitor):
            runner.main()
    elif args.action == "generate":
        from wmbench_neural import generate
        original_channel = generate.NeuralChannel

        def channel(engine, cache):
            instance = original_channel(engine, cache)
            instance.identity = reconcile_identity(settings["results"], engine, instance.identity)
            return instance

        with patch.object(generate, "NeuralChannel", channel):
            runner.main()
    else:
        runner.main()


if __name__ == "__main__":
    main()
'''
REPAIR_SHA256 = '94bb725be25d6f550284c75ee76051674b92a5dddd8922eda956b8895aefa383'
REPAIR_PATH = PROJECT / "src/neural_sdxl_identity_fix.py"
repair_bytes = REPAIR_SOURCE.encode("utf-8")
if hashlib.sha256(repair_bytes).hexdigest() != REPAIR_SHA256:
    raise ValueError("Repair source checksum mismatch")
if REPAIR_PATH.exists() and REPAIR_PATH.read_bytes() != repair_bytes:
    raise ValueError("Different recovery overlay exists. Use a new scratch folder; keep OUTPUT_ROOT.")
REPAIR_PATH.parent.mkdir(parents=True, exist_ok=True)
REPAIR_PATH.write_bytes(repair_bytes)
if hashlib.sha256(REPAIR_PATH.read_bytes()).hexdigest() != REPAIR_SHA256:
    raise ValueError("Repair readback failed")
ENV["WMBENCH_NEURAL_IDENTITY_FIX_SHA256"] = REPAIR_SHA256
print("Recovery ready: only Diffusers bookkeeping-key order can differ.")
print("Existing results, frozen execution identities and original source ZIP are preserved.")


## 1b. Restore the exact VideoSeal checkpoint

Adds one readable transport overlay. The saved bank supplies the expected hash; the decoder, weights, attack grid and clean controls are not changed. The verified 218 MiB checkpoint and receipts live under `OUTPUT_ROOT/_repairs/videoseal-download-v1/`, outside temporary model slots.


In [ ]:
# Transport-only VideoSeal recovery; the frozen release is not edited.
VIDEOSEAL_REPAIR_SOURCE = r'''"""Restore the exact saved VideoSeal checkpoint before the frozen prepare step.

Only the checkpoint transport changes. Model code, options, dependencies and
all original evidence checks are delegated to the unchanged release. The SDXL
identity overlay remains in place, including its worker propagation.
"""

import argparse
import hashlib
import http.client
import json
import os
from pathlib import Path
import shutil
import sys
import tempfile
import time
import urllib.error
import urllib.request
from unittest.mock import patch


VERSION = "videoseal-download-v1"
ENTRYPOINT = "neural_videoseal_download_fix"
HASH_ENV = "WMBENCH_VIDEOSEAL_DOWNLOAD_FIX_SHA256"
BASE_ZIP_SHA256 = "5b70653cf2fb7b23d6ff1b8585aa1b684808ba311d956cf5fc20da4b6a6311e2"
IDENTITY_FIX_SHA256 = "94bb725be25d6f550284c75ee76051674b92a5dddd8922eda956b8895aefa383"
WIBE_REVISION = "acf94ed81db4243d22fb7a7e075aff1671bc58e0"
CHECKPOINT_KEY = "model_files/videoseal/y_256b_img.pth"
CHECKPOINT_SHA256 = "3d2ff2523d2a89e3532c6dfdcf693098799326e9b3e74c185e815e9baa8340a3"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/videoseal/y_256b_img.pth"
MIRROR_URL = ("https://huggingface.co/Svetozar1993/videoseal-1.0/resolve/"
    "35116c2e949486b08c2632df4ad8f73f9b177b55/y_256b_img.pth")
CHUNK_BYTES = 1024 * 1024


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(CHUNK_BYTES), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verified(path, expected):
    """Never replace an existing file whose provenance cannot be established."""
    path = Path(path)
    if path.is_symlink():
        raise ValueError(f"Checkpoint must not be a symlink: {path}")
    if not path.exists():
        return False
    if not path.is_file() or file_sha256(path) != expected:
        raise ValueError(f"Checkpoint SHA-256 mismatch; file preserved: {path}")
    return True


def publish(source, target, expected):
    """Verified copy with a same-directory temporary file and readback check."""
    source, target = Path(source), Path(target)
    if verified(target, expected):
        return
    if not verified(source, expected):
        raise FileNotFoundError(source)
    target.parent.mkdir(parents=True, exist_ok=True)
    descriptor, name = tempfile.mkstemp(prefix=".videoseal-", suffix=".part", dir=target.parent)
    temporary = Path(name)
    try:
        with os.fdopen(descriptor, "wb") as output, source.open("rb") as incoming:
            shutil.copyfileobj(incoming, output, CHUNK_BYTES)
            output.flush()
            os.fsync(output.fileno())
        verified(temporary, expected)
        # The original controller lease serializes jobs. Recheck nevertheless;
        # a conflicting pre-existing file must never be overwritten.
        if not verified(target, expected):
            os.replace(temporary, target)
        verified(target, expected)
    finally:
        temporary.unlink(missing_ok=True)  # Only this call's private partial file.


def fetch_checkpoint(target, expected, deadline, attempts=3):
    """Try Meta then a pinned byte-identical mirror, always enforcing bank SHA-256."""
    target = Path(target)
    if verified(target, expected):
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, attempts + 1):
        url = CHECKPOINT_URL if attempt == 1 else MIRROR_URL
        remaining = deadline - time.time()
        if remaining <= 0:
            raise TimeoutError("Session budget reached before VideoSeal download")
        descriptor, name = tempfile.mkstemp(prefix=".videoseal-download-", suffix=".part", dir=target.parent)
        temporary = Path(name)
        try:
            print(f"VIDEOSEAL DOWNLOAD {attempt}/{attempts}: {url}", flush=True)
            request = urllib.request.Request(url, headers={"User-Agent": "wmbench-videoseal-recovery/1"})
            with os.fdopen(descriptor, "wb") as output:
                with urllib.request.urlopen(request, timeout=min(60, remaining)) as response:
                    total = response.headers.get("Content-Length")
                    total = int(total) if total is not None else None
                    count, reported = 0, 0
                    while True:
                        if time.time() >= deadline:
                            raise TimeoutError("Session budget reached during VideoSeal download")
                        chunk = response.read(CHUNK_BYTES)
                        if not chunk:
                            break
                        output.write(chunk)
                        count += len(chunk)
                        if count - reported >= 16 * CHUNK_BYTES:
                            print(f"VIDEOSEAL DOWNLOAD: {count / CHUNK_BYTES:.0f} MiB received", flush=True)
                            reported = count
                    if total is not None and count != total:
                        raise OSError(f"Truncated checkpoint: {count} of {total} bytes")
                output.flush()
                os.fsync(output.fileno())
            # A wrong complete response is not retried or accepted as new weights.
            verified(temporary, expected)
            publish(temporary, target, expected)
            return
        except (OSError, urllib.error.URLError, http.client.HTTPException) as error:
            last_error = error
            print(f"VIDEOSEAL download interrupted ({type(error).__name__}); results are unchanged.", flush=True)
        finally:
            temporary.unlink(missing_ok=True)
    raise RuntimeError(
        "VideoSeal checkpoint download failed at Meta and the pinned mirror. Rerun with the same output folder. "
        f"Alternatively place the exact SHA-256 {expected} checkpoint at {target}. "
        "No weights or results were substituted."
    ) from last_error


def expected_checkpoint(job):
    """Require the original saved bank, not a newly chosen model/checkpoint."""
    path = Path(job["bank"]) / "videoseal/_meta/model_identity.json"
    saved = json.loads(path.read_text(encoding="utf-8"))
    if (saved.get("registry") != job["model"]
            or job["model"].get("source", {}).get("revision") != WIBE_REVISION):
        raise ValueError("VideoSeal registry/source differs from the saved bank; recovery refused")
    if saved.get("weights", {}).get(CHECKPOINT_KEY) != CHECKPOINT_SHA256:
        raise ValueError("Saved bank does not use the audited VideoSeal checkpoint; recovery refused")
    return path


def ensure_videoseal_checkpoint(job):
    """Preseed WIBE's existing required-file path; do not patch its adapter."""
    if job.get("model", {}).get("id") != "videoseal":
        return None
    session = str(job["session"])
    if not session.isdecimal():
        raise ValueError("Invalid recovery session identifier")
    identity_path = expected_checkpoint(job)
    root = Path(job["results"]) / "_repairs" / VERSION
    cache = root / "assets" / CHECKPOINT_SHA256 / "y_256b_img.pth"
    local = Path(job["slot"]) / "assets/videoseal" / CHECKPOINT_KEY
    # Check both existing locations before doing any copy/download.
    has_local = verified(local, CHECKPOINT_SHA256)
    has_cache = verified(cache, CHECKPOINT_SHA256)
    source = "drive-cache" if has_cache else "existing-slot" if has_local else "verified-download"
    if not has_cache:
        if has_local:
            publish(local, cache, CHECKPOINT_SHA256)
        else:
            fetch_checkpoint(cache, CHECKPOINT_SHA256, job["deadline"])
    publish(cache, local, CHECKPOINT_SHA256)
    from wmbench_full_payload.storage import freeze

    # This audit is outside model assets and outside scientific identities.
    freeze(root / "receipts" / f"{session}.json", dict(
        repair=VERSION, repair_sha256=file_sha256(__file__),
        base_zip_sha256=BASE_ZIP_SHA256, identity_fix_sha256=IDENTITY_FIX_SHA256,
        checkpoint_sha256=CHECKPOINT_SHA256, checkpoint_key=CHECKPOINT_KEY,
        permitted_download_urls=[CHECKPOINT_URL, MIRROR_URL], bank_identity_sha256=file_sha256(identity_path),
        checkpoint_size=local.stat().st_size,
        scope="transport only; original model/checkpoint/dependency/Clean checks remain enabled",
    ))
    print(f"VIDEOSEAL CHECKPOINT VERIFIED ({source}): {CHECKPOINT_SHA256}", flush=True)
    print(f"VIDEOSEAL durable cache: {cache}", flush=True)
    return local


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("action", choices=("run", "validate", "prepare", "quality",
        "geometry", "controls", "generate", "decode", "report"))
    parser.add_argument("--settings", type=Path, required=True)
    args = parser.parse_args()
    actual = file_sha256(__file__)
    if os.environ.get(HASH_ENV) != actual:
        raise RuntimeError("VideoSeal recovery checksum absent or changed; rerun the repair cell")
    import neural_sdxl_identity_fix as identity_fix

    if (identity_fix.module_sha256() != IDENTITY_FIX_SHA256
            or os.environ.get(identity_fix.HASH_ENV) != IDENTITY_FIX_SHA256):
        raise RuntimeError("Original identity-recovery module/hash changed; recovery refused")
    if args.action == "prepare":
        ensure_videoseal_checkpoint(json.loads(args.settings.read_text(encoding="utf-8")))
    # Existing forwarding adds its own checksum; the new checksum must also be
    # explicit in decoder environments, which may not inherit os.environ.
    original_redirect = identity_fix.redirect_child

    def redirect(command, env, expected_sha256):
        forwarded, child_env = original_redirect(command, env, expected_sha256)
        if forwarded is not command:
            child_env = {**child_env, HASH_ENV: actual}
        return forwarded, child_env

    with patch.object(identity_fix, "ENTRYPOINT", ENTRYPOINT), \
            patch.object(identity_fix, "redirect_child", redirect):
        identity_fix.main()


if __name__ == "__main__":
    main()
'''
VIDEOSEAL_REPAIR_SHA256 = '11cf3a99bac14c53c5f87ee4d3624bcb1d833867f5d8292b719910712afda674'
VIDEOSEAL_REPAIR_PATH = PROJECT / "src/neural_videoseal_download_fix.py"
videoseal_repair_bytes = VIDEOSEAL_REPAIR_SOURCE.encode("utf-8")
if hashlib.sha256(videoseal_repair_bytes).hexdigest() != VIDEOSEAL_REPAIR_SHA256:
    raise ValueError("VideoSeal repair source checksum mismatch")
if VIDEOSEAL_REPAIR_PATH.exists() and VIDEOSEAL_REPAIR_PATH.read_bytes() != videoseal_repair_bytes:
    raise ValueError("Different VideoSeal overlay exists; use new scratch, keep OUTPUT_ROOT.")
VIDEOSEAL_REPAIR_PATH.parent.mkdir(parents=True, exist_ok=True)
VIDEOSEAL_REPAIR_PATH.write_bytes(videoseal_repair_bytes)
if hashlib.sha256(VIDEOSEAL_REPAIR_PATH.read_bytes()).hexdigest() != VIDEOSEAL_REPAIR_SHA256:
    raise ValueError("VideoSeal repair readback failed")
ENV["WMBENCH_VIDEOSEAL_DOWNLOAD_FIX_SHA256"] = VIDEOSEAL_REPAIR_SHA256
print("VideoSeal transport recovery ready. Exact saved checkpoint SHA-256 required.")
print("Existing results and pending attack packages are preserved. No GPU job started yet.")


## 2. System libraries

Restore the already reviewed MagickWand/GL dependencies for author implementations.


In [ ]:
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)


## 3. Validate bank and freeze captions/seeds

No images are re-embedded. A missing bank/caption or changed release fails explicitly.


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "neural_videoseal_download_fix", "validate", "--settings", str(CONFIG)], env=ENV, check=True)


## 4. Run

Clean preflight → GPU attack batches → saved ZIPs → unchanged decoder → durable scores → delete ZIPs. Repeat per pipeline and method. CPU input preparation/PNG encoding/Drive writes overlap GPU batches.

Temporary Drive space is needed for one method/pipeline stage (potentially tens of GiB). Logs and completed results survive disconnects. A failed call stops safely; restart the same cells.


In [ ]:
command = [str(CONTROLLER), "-m", "neural_videoseal_download_fix", "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", start_new_session=True)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Run stopped: inspect persistent logs. Completed rows and pending ZIPs are safe.")
except BaseException:
    if process.poll() is None:
        import signal
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    raise


## 5. Report

Rebuilding reports performs no inference. Missing or safety-blocked generations are not decoding errors. The additional resize-only diagnostic is excluded from the 21-condition experiment.


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "neural_videoseal_download_fix", "report", "--settings", str(CONFIG)], env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))
print("Aggregate report:", OUTPUT_ROOT / "_reports")
print("Per-method tables:", OUTPUT_ROOT / "<method>/tables")
print("Decoded bits and task provenance:", OUTPUT_ROOT / "<method>/records")


## Interpretation and limits

The canary has up to **60,900** evaluations; N=100 has up to **1,218,000**, fewer when a matched carrier is unavailable. A fixed batch improves throughput, not the scientific task count. This canary itself may span multiple sessions.

Static and CPU smoke checks are recorded in VALIDATION.md. Real CUDA pipelines still need this Colab acceptance run; no full-GPU success or runtime estimate is invented. Protocol source: bundled committed revision; live GitHub access was unavailable during preparation.
